# ENGRAMA V3 × TinyStories — Modelo autoregresivo de ~20M con GPT-2 🧠⚡

Entrenamiento **completo** de un modelo de lenguaje autoregresivo **sin atención** (cero $QK^T$, cero softmax temporal) con la librería [ENGRAMA V3](https://github.com/bueormnew/engrama) instalada desde GitHub.

| Aspecto | Valor |
|---|---|
| Arquitectura | ENGRAMA V3 (sinapsis factorizadas, offsets diádicos, caché jerárquico) |
| Parámetros | ~20.3M (config `L=9`, cobertura binaria completa) |
| Datos | **TinyStories completo** (train + valid, Hugging Face) |
| Tokenizer | **GPT-2 BPE** (vocabulario 50,257) |
| Contexto | **512 tokens** |
| Salida | checkpoint `engrama_v3_20m_gpt2/` + muestras de inferencia |

> 🔧 **FAST_MODE = False** → entrenamiento real (recomendado: GPU T4 o mejor).
> **FAST_MODE = True** → smoke test de todo el pipeline en minutos, incluso en CPU.

- Autor: **BUEORM** · Licencia: **AGPL-3.0**


## 1️⃣ Instalación (ENGRAMA V3 desde GitHub + transformers)

In [ ]:
# ENGRAMA se instala SIEMPRE desde GitHub (el nombre 'engrama' en PyPI es otro paquete)
!pip install -q git+https://github.com/bueormnew/engrama.git transformers

import os
import math
import time
import random

import torch
import torch.nn.functional as F

import engrama
from engrama import EngramaConfig, EngramaModel, Generator, save_model, load_model

print('ENGRAMA', engrama.__version__, '| torch', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(CPU)')


## 2️⃣ Configuración central

Todos los hiperparámetros en un solo lugar. En **FULL** (`FAST_MODE=False`) se 
construye el modelo real de ~20.3M con contexto 512; en **FAST** se reducen
modelo, datos y pasos solo para validar el pipeline de punta a punta.


In [ ]:
# ------------------------- MODO -----------------------------------------
FAST_MODE = True   # False => entrenamiento REAL ~20M / TinyStories completo / 512
SEED = 1234

# ------------------------- DATOS ----------------------------------------
SEQ_LEN = 512                 # contexto del modelo real (tokens GPT-2)
MAX_TRAIN_SEQS = 80_000       # secuencias de entrenamiento (~41M tokens)
MAX_VALID_SEQS = 1_000        # secuencias de validación

# ------------------------- MODELO (~20.3M en FULL) ----------------------
MODEL_KW = dict(
    d_model=256, d_gate=32, d_ff=1024,
    num_cells=8, num_encoder_layers=2,
    num_consolidation_layers=9,   # L >= ceil(log2 512) => cobertura binaria completa
    num_candidates=4, candidate_aggregation='logsumexp',
    synapse_rank=32, version='v3', global_anchor=False,
)

# ------------------------- ENTRENAMIENTO --------------------------------
# batch 16: logits (B x 512 x 50257) en fp32 ocupan ~1.7 GB; batch 32 arriesga OOM
# en una T4 de 16 GB durante el backward de cross-entropy.
BATCH_SIZE = 16
EPOCHS = 1
LR = 6e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 200
LOG_EVERY = 50
EVAL_EVERY = 250
SAMPLE_EVERY = 500
EVAL_BATCHES = 25

if FAST_MODE:
    SEQ_LEN = 128
    MAX_TRAIN_SEQS = 256
    MAX_VALID_SEQS = 64
    MODEL_KW.update(d_model=128, d_gate=16, d_ff=512, num_cells=4,
                    num_encoder_layers=2, num_consolidation_layers=7,
                    num_candidates=2, synapse_rank=16)
    BATCH_SIZE, EPOCHS = 8, 2
    WARMUP_STEPS = 10
    LOG_EVERY, EVAL_EVERY, SAMPLE_EVERY, EVAL_BATCHES = 10, 32, 32, 4

SAVE_DIR = ('/kaggle/working/engrama_v3_20m_gpt2'
            if os.path.isdir('/kaggle/working') else './engrama_v3_20m_gpt2')
random.seed(SEED); torch.manual_seed(SEED)
print(f'Modo={"FAST" if FAST_MODE else "FULL"} | SEQ_LEN={SEQ_LEN} | '
f'train_seqs<={MAX_TRAIN_SEQS} | ckpt -> {SAVE_DIR}')


## 3️⃣ Datos: TinyStories completo

Descarga `TinyStoriesV2-GPT4-train.txt` / `-valid.txt` de Hugging Face (con fallbacks: dataset montado en Kaggle → corpus sintético offline). La lectura es **streaming por cuentos** para no cargar ~2 GB en RAM.


In [ ]:
TRAIN_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-train.txt')
VALID_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-valid.txt')

def download(url, path):
    import urllib.request
    if not os.path.exists(path):
        print('Descargando', url, '...')
        urllib.request.urlretrieve(url, path)
    return path

def iter_stories(path):
    """Generador streaming: produce cuentos separados por línea en blanco."""
    buf = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.strip():
                buf.append(line.rstrip('\n'))
            elif buf:
                yield ' '.join(buf).strip()
                buf = []
    if buf:
        yield ' '.join(buf).strip()

FALLBACK_STORY = (
    'Once upon a time there was a little cat named Lily. Lily liked to play in the '
    'garden with her red ball. One day a dog named Tom came and they played together '
    'all day. At night Lily went home, ate her dinner and slept. The end.'
)

train_path = valid_path = None
try:
    if FAST_MODE:
        raise RuntimeError('FAST_MODE: solo valid.txt')
    train_path = download(TRAIN_URL, 'tinystories_train.txt')
    valid_path = download(VALID_URL, 'tinystories_valid.txt')
except Exception as exc:
    try:
        valid_path = download(VALID_URL, 'tinystories_valid.txt')
        train_path = train_path or valid_path
        print('Aviso: usando valid split para entrenar (%s)' % type(exc).__name__)
    except Exception as exc2:
        print('Sin internet (%s). Corpus sintético offline.' % type(exc2).__name__)
        with open('tinystories_synth.txt', 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(FALLBACK_STORY for _ in range(3000)))
        train_path = valid_path = 'tinystories_synth.txt'

print('train:', train_path, '| valid:', valid_path)


## 4️⃣ Tokenizer GPT-2 (BPE, vocabulario 50,257)

Adaptador con la interfaz exacta que espera `engrama.Generator` (`encode/decode/SPECIAL_TOKENS/vocab_size`). Sin internet, cae al tokenizer de caracteres de ENGRAMA para que el notebook siempre sea ejecutable.


In [ ]:
class GPT2Adapter:
    """Interfaz ENGRAMA sobre el tokenizer GPT-2 de Hugging Face."""
    def __init__(self, hf_tok):
        self.tok = hf_tok
        eot = hf_tok.eos_token_id  # 50256 <|endoftext|>
        self.SPECIAL_TOKENS = {'<eos>': eot, '<bos>': eot, '<pad>': eot}
        self.vocab_size = len(hf_tok)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = list(self.tok.encode(text, add_special_tokens=False))
        if add_bos:
            ids = [self.SPECIAL_TOKENS['<bos>']] + ids
        if add_eos:
            ids = ids + [self.SPECIAL_TOKENS['<eos>']]
        return ids

    def decode(self, ids, skip_special_tokens=True):
        return self.tok.decode(list(ids), skip_special_tokens=skip_special_tokens)

    def encode_batch(self, texts):
        outs = self.tok(list(texts), add_special_tokens=False)['input_ids']
        return [list(ids) + [self.SPECIAL_TOKENS['<eos>']] for ids in outs]

tokenizer = None
try:
    os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '10')
    from transformers import GPT2TokenizerFast
    _hf = GPT2TokenizerFast.from_pretrained('gpt2')
    # sanity check: sin internet, from_pretrained puede devolver un stub vacío
    if len(_hf) < 1000 or not _hf.encode('hello world', add_special_tokens=False):
        raise RuntimeError(f'GPT-2 incompleto (vocab={len(_hf)})')
    tokenizer = GPT2Adapter(_hf)
    print('Tokenizer: GPT-2 BPE, vocab =', tokenizer.vocab_size)
except Exception as exc:
    from engrama import EngramaTokenizer
    print('GPT-2 no disponible (%s) -> fallback char-level' % type(exc).__name__)
    sample = '\n'.join(s for _, s in zip(range(200), iter_stories(train_path)))
    tokenizer = EngramaTokenizer().fit_on_text(sample)
    print('Tokenizer: char-level fallback, vocab =', tokenizer.vocab_size)

VOCAB_SIZE = tokenizer.vocab_size
EOS_ID = tokenizer.SPECIAL_TOKENS['<eos>']
print(repr(tokenizer.decode(tokenizer.encode('Once upon a time, there was a cat.')[:12])))


## 5️⃣ Tokenización y secuencias de 512 tokens

Cada cuento se tokeniza y se separa con `<|endoftext|>`; el stream plano se corta
en ventanas de `SEQ_LEN + 1` (entrada + objetivo desplazado, LM autoregresivo).


In [ ]:
def stories_to_ids(path, max_stories=None, max_ids=None):
    ids = []
    stories = iter_stories(path)
    batch, n = [], 0
    while True:
        story = next(stories, None)
        if story is None and not batch:
            break
        if story is not None:
            batch.append(story)
        if story is None or len(batch) >= 256:
            if hasattr(tokenizer, 'encode_batch'):
                batch_ids = tokenizer.encode_batch(batch)
            else:
                batch_ids = [tokenizer.encode(t, add_eos=True) for t in batch]
            for ids_i in batch_ids:
                ids.extend(ids_i)
            n += len(batch)
            batch = []
            if n % 5000 < 256:
                print(f'  {n:,} cuentos | {len(ids):,} tokens ...')
            if (max_stories and n >= max_stories) or \
               (max_ids and len(ids) >= max_ids):
                break
        if story is None:
            break
    return ids[:max_ids] if max_ids else ids

def make_xy(ids, seq_len, max_seqs):
    chunk = seq_len + 1
    n = min(max_seqs, len(ids) // chunk)
    xs, ys = [], []
    for i in range(n):
        w = ids[i * chunk:(i + 1) * chunk]
        xs.append(torch.tensor(w[:-1], dtype=torch.long))
        ys.append(torch.tensor(w[1:], dtype=torch.long))
    return torch.stack(xs), torch.stack(ys)

t0 = time.time()
print('Tokenizando split de entrenamiento ...')
train_ids = stories_to_ids(train_path, max_ids=MAX_TRAIN_SEQS * (SEQ_LEN + 1))
print('Tokenizando split de validación ...')
valid_ids = stories_to_ids(valid_path,
                           max_ids=MAX_VALID_SEQS * (SEQ_LEN + 1) + 2048)

Xtr, Ytr = make_xy(train_ids, SEQ_LEN, MAX_TRAIN_SEQS)
Xva, Yva = make_xy(valid_ids, SEQ_LEN, MAX_VALID_SEQS)
print(f'train: {Xtr.shape[0]:,} secuencias x {SEQ_LEN} tokens '
      f'({Xtr.numel():,} tokens) | valid: {Xva.shape[0]:,} | '
      f'{time.time() - t0:.1f}s')
assert Xtr.shape[0] > 0 and Xva.shape[0] > 0, 'datos insuficientes'


## 6️⃣ Construcción del modelo (~20.3M, contexto 512, cobertura binaria completa)


In [ ]:
config = EngramaConfig(
    vocab_size=VOCAB_SIZE, context_length=SEQ_LEN, **MODEL_KW)
torch.manual_seed(SEED)
model = EngramaModel(config).to(DEVICE)

n_params = model.num_parameters()
rf = config.receptive_field()
print(f'Parámetros: {n_params:,}  (~{n_params/1e6:.1f}M)')
print(f'Campo receptivo: {rf["max_reach"]} tokens | cubre contexto: {rf["covers_context"]}')
print('Horizontes de caché jerárquico:', config.cache_horizons())
print('Offsets por capa:', [config.get_layer_offsets(l)
                          for l in range(config.num_consolidation_layers)])


## 7️⃣ Entrenamiento

AdamW + clipping + warmup lineal + decaimiento coseno, con evaluación periódica
, muestreo de generación y **guardado del mejor checkpoint por loss de validación**.


In [ ]:
train_ds = torch.utils.data.TensorDataset(Xtr, Ytr)
valid_ds = torch.utils.data.TensorDataset(Xva, Yva)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE,
                                       shuffle=True, drop_last=True)
valid_dl = torch.utils.data.DataLoader(valid_ds, batch_size=BATCH_SIZE)
TOTAL_STEPS = len(train_dl) * EPOCHS
print(f'{TOTAL_STEPS} pasos totales ({len(train_dl)}/época x {EPOCHS})')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

@torch.no_grad()
def evaluate(max_batches=EVAL_BATCHES):
    model.eval()
    total, nb = 0.0, 0
    for i, (xb, yb) in enumerate(valid_dl):
        if i >= max_batches:
            break
        logits = model(xb.to(DEVICE))
        total += F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                                 yb.to(DEVICE).reshape(-1)).item()
        nb += 1
    model.train()
    return total / max(1, nb)

generator = Generator(model, tokenizer)
history, best_val = [], float('inf')
global_step, seen = 0, 0
t0 = time.time()
model.train()
for epoch in range(EPOCHS):
    for xb, yb in train_dl:
        lr = lr_at(global_step)
        for gparam in optimizer.param_groups:
            gparam['lr'] = lr
        logits = model(xb.to(DEVICE))
        loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE),
                               yb.to(DEVICE).reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        history.append((global_step, loss.item()))

        if global_step % LOG_EVERY == 0:
            print(f'paso {global_step:>5}/{TOTAL_STEPS} | loss {loss.item():.4f} | '
              f'lr {lr:.2e} | {time.time() - t0:.0f}s')
        if (global_step + 1) % EVAL_EVERY == 0 or global_step + 1 == TOTAL_STEPS:
            val = evaluate()
            print(f'  [eval] paso {global_step + 1}: val_loss {val:.4f} | '
                  f'val_ppl {math.exp(min(20, val)):.2f}')
            if val < best_val:
                best_val = val
                save_model(model, SAVE_DIR)
                print(f'  [ckpt] mejor checkpoint guardado (val {best_val:.4f})')
        if (global_step + 1) % SAMPLE_EVERY == 0:
            model.eval()
            print('  [muestra]', repr(generator.generate(
                'Once upon a time', max_new_tokens=64, temperature=0.8,
                top_k=40, stop_at_eos=True)))
            model.train()
        global_step += 1

print(f'Entrenamiento terminado en {time.time() - t0:.1f}s | '
      f'mejor val_loss {best_val:.4f}')


### Curva de pérdida

In [ ]:
try:
    import matplotlib.pyplot as plt
    steps, losses = zip(*history)
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, alpha=0.3, label='train loss')
    w = max(5, len(losses) // 50)
    plt.plot(steps[w - 1:], [sum(losses[i - w:i]) / w for i in range(w, len(losses) + 1)],
             label=f'media móvil ({w})')
    plt.xlabel('paso'); plt.ylabel('cross-entropy'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('ENGRAMA V3 ~20M en TinyStories (GPT-2, seq 512)')
    plt.show()
except ImportError:
    print('matplotlib no disponible; historial:', history[:3], '...', history[-3:])


## 8️⃣ Evaluación final y guardado


In [ ]:
final_val = evaluate(max_batches=len(valid_dl))
print(f'Validación completa: loss {final_val:.4f} | '
      f'perplejidad {math.exp(min(20, final_val)):.2f}')

if final_val < best_val:
    save_model(model, SAVE_DIR)
    best_val = final_val
print('Checkpoint final en', SAVE_DIR, ':', sorted(os.listdir(SAVE_DIR)))

import json as _json
with open(os.path.join(SAVE_DIR, 'training_log.json'), 'w') as f:
    _json.dump({'params': n_params, 'seq_len': SEQ_LEN, 'vocab': VOCAB_SIZE,
                'steps': TOTAL_STEPS, 'best_val_loss': best_val,
                'fast_mode': FAST_MODE, 'seed': SEED}, f, indent=2)
print('training_log.json escrito')


## 9️⃣ Inferencia con el modelo entrenado

Recarga del checkpoint desde disco (prueba de persistencia completa) y generación
con temperatura / top-k / top-p y parada en `<|endoftext|>`.


In [ ]:
loaded_model, _ = load_model(SAVE_DIR, device=DEVICE)
loaded_model.eval()
gen = Generator(loaded_model, tokenizer)
print('Checkpoint recargado:', f'{loaded_model.num_parameters():,}', 'parámetros\n')

for prompt in ['Once upon a time', 'One day, a little girl named Anna',
               'Tom found a big red ball']:
    out = gen.generate(prompt, max_new_tokens=120, temperature=0.8,
                       top_k=40, top_p=0.95, stop_at_eos=True)
    print('==>', prompt)
    print(out.replace('<|endoftext|>', '').strip()[:600], '\n')


### Invarianza causal del checkpoint (verificación rápida)

In [ ]:
x = Xva[:2, :64].to(DEVICE)
with torch.no_grad():
    full = loaded_model(x)
    for mode in ('full', 'hierarchical'):
        cache = loaded_model.get_cache(N_max=64, mode=mode)
        steps = [loaded_model.step_forward(x[:, t:t + 1], cache, t)[0]
                 for t in range(64)]
        inc = torch.stack(steps, dim=1)
        print(f'caché {mode:13s}: max |diff| = '
              f'{(full - inc).abs().max().item():.2e}  (< 1e-4 OK)')


## ✅ Notas honestas

- El entrenamiento **real** es `FAST_MODE = False`: TinyStories completo, seq 512,
  ~41M tokens vistos, 3000+ pasos. Con FAST_MODE solo se valida el pipeline; el
  texto generado será incipiente.
- Calidad esperada en FULL: TinyStories es un benchmark amable para modelos de
  ~20M (perplejidad de validación típicamente < 10 con datos suficientes); el
  objetivo aquí es demostrar el pipeline completo de ENGRAMA V3 end-to-end, no
  batir el estado del arte.
- Reproducibilidad: `training_log.json` en el checkpoint registra semilla, pasos,
  pérdidas y configuración.
- Licencia AGPL-3.0 · Autor: BUEORM · [github.com/bueormnew/engrama](https://github.com/bueormnew/engrama)
